# 用 LoRA 微調 LLM (SFT with TRL + PEFT)

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/06-transformers/05-LoRA-SFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
!pip install -q -U transformers peft trl
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/06-transformers/'
for fn in ['zh_numbers.py', 'q_lora_params.py', 'q_upper.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

大型模型全部參數一起微調，需要很多 GPU 記憶體（參數 + 梯度 + optimizer 狀態，大約是模型大小的 4 倍以上）。

**LoRA (Low-Rank Adaptation)**：凍結原本的權重 $W$，只訓練一個低秩 (low-rank) 的修正量

$$W' = W + \frac{\alpha}{r} BA, \quad B\in\mathbb{R}^{d\times r}, A\in\mathbb{R}^{r\times k}, r \ll d$$

可訓練參數通常不到 1%，T4 上也能微調。

這個單元的任務：教 Qwen3-0.6B 把**中文數字轉成阿拉伯數字**（例如「三千零五十二」→ 3052）。
這看起來簡單，但小模型其實常常算錯。資料用程式產生（`zh_numbers.py`），所以要多少有多少，也很容易評估準確率。

In [ ]:
import random, time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from zh_numbers import num2zh, num2zh_upper, random_number
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
for n in [7, 15, 105, 3052, 10050, 20000000, 10100100]:
    print(n, num2zh(n))

## 準備資料

TRL 的 `SFTTrainer` 支援 prompt-completion 格式的對話資料，預設只對 completion（模型的回答）計算 loss。

In [ ]:
from datasets import Dataset
PROMPT = "把下面的中文數字轉成阿拉伯數字，只輸出數字：{}"

def make_example(n, to_zh=num2zh):
    return {"prompt": [{"role": "user", "content": PROMPT.format(to_zh(n))}],
            "completion": [{"role": "assistant", "content": str(n)}],
            "answer": n}

rng = random.Random(42)
test_numbers = [random_number(rng) for _ in range(200)]
train_numbers = [n for n in (random_number(rng) for _ in range(6000)) if n not in set(test_numbers)][:4000]
train_ds = Dataset.from_list([make_example(n) for n in train_numbers])
test_ds = Dataset.from_list([make_example(n) for n in test_numbers])
train_ds[0]

## 訓練前：原本的模型表現如何？

In [ ]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")

In [ ]:
@torch.no_grad()
def evaluate(model, ds, batch_size=50, show=5):
    tokenizer.padding_side = "left"
    model.eval()
    correct, outputs = 0, []
    for i in range(0, len(ds), batch_size):
        batch = ds[i:i + batch_size]
        texts = [tokenizer.apply_chat_template(p, tokenize=False, add_generation_prompt=True, enable_thinking=False)
                 for p in batch["prompt"]]
        inputs = tokenizer(texts, padding=True, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False)
        for o, ans in zip(out, batch["answer"]):
            pred = tokenizer.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().replace(",", "")
            correct += pred == str(ans)
            outputs.append((ans, pred))
    for ans, pred in outputs[:show]:
        print(f"{num2zh(ans)} → {pred}  {'✅' if pred == str(ans) else '❌ ' + str(ans)}")
    return correct / len(ds)

print("accuracy:", evaluate(model, test_ds))

## 加上 LoRA

`target_modules="all-linear"`：在所有線性層 (attention 的 q, k, v, o 和 MLP) 加上 LoRA。

In [ ]:
from peft import LoraConfig
peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")

## 用 TRL 的 SFTTrainer 訓練

T4 上的注意事項：
* 模型以 fp16 載入，`fp16=True`、`bf16=False`（SFTConfig 預設會開 bf16，T4 不支援）
* LoRA 的參數會自動用 fp32 訓練，搭配 GradScaler，數值比較穩定

In [ ]:
from trl import SFTConfig, SFTTrainer
tokenizer.padding_side = "right"
args = SFTConfig(
    output_dir="qwen3-zh-number-lora",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    logging_steps=20,
    max_length=128,
    fp16=True,
    bf16=False,
    save_strategy="no",
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds.remove_columns("answer"),
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

In [ ]:
# 看一下實際送進模型的資料：completion_mask 為 1 的部份才計算 loss
ex = trainer.train_dataset[0]
print(tokenizer.decode(ex["input_ids"]))
print("計算 loss 的部份:", repr(tokenizer.decode([t for t, m in zip(ex["input_ids"], ex["completion_mask"]) if m])))

In [ ]:
t0 = time.perf_counter()
trainer.train()
print(f"訓練時間 {time.perf_counter() - t0:.0f}s")

## 訓練後

In [ ]:
print("accuracy:", evaluate(trainer.model, test_ds, show=10))

## Q
LoRA 只訓練了多少比例的參數？請自己數一數 `trainer.model` 中 `requires_grad=True` 的參數，並列出前幾個參數的名稱和形狀。

In [ ]:
# %load q_lora_params.py
%run -i q_lora_params.py

## 存檔、合併

LoRA adapter 很小（幾 MB 到幾十 MB），可以單獨存檔、分享。也可以把 $BA$ 合併回原本的權重，變成一個一般的模型。

In [ ]:
trainer.save_model("qwen3-zh-number-lora")       # 只存 adapter
!ls -lh qwen3-zh-number-lora

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("qwen3-zh-number-merged")
tokenizer.save_pretrained("qwen3-zh-number-merged")

In [ ]:
# 合併後的模型就是普通的 transformers 模型，模型本身其他的能力還在嗎？
messages = [{"role": "user", "content": "請用一句話介紹台灣。"}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                       return_dict=True, return_tensors="pt").to(device)
out = merged.generate(**inputs, max_new_tokens=100, do_sample=False)
print(tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))

## Q
目前的模型只看過一般的中文數字。用 `num2zh_upper`（大寫數字，例如「參仟零伍拾貳」）建立測試資料：
1. 現在的模型對大寫數字的準確率是多少？
2. 把大寫數字的資料加入訓練，重新訓練一個 LoRA，兩種寫法都能正確轉換嗎？

In [ ]:
# %load q_upper.py
%run -i q_upper.py